# ⛳ Mini-Golf in the Wind

Welcome to the windiest mini-golf course in the world!

## The Story

You find yourself on a peculiar mini-golf hole. The green stretches out before you — but the hole is **invisible**, hidden under a patch of artificial turf. All you can do is:

1. 🏌️ Choose a **putting force** (0 = barely a tap, 100 = full swing)
2. 🌬️ **Putt** — the wind shifts your ball by a random amount every time
3. 📢 Hear the caddie announce: *"3.2 feet PAST the hole"* or *"1.8 feet SHORT"*

The caddie always tells you **how far** and **in which direction** you missed — but never where the hole actually is.

### The Challenge

Your goal is to find the right putting force to sink the ball, using only the caddie's signed feedback over many putts.

This is the **Robbins-Monro problem**: finding an unknown fixed point using noisy, sequential observations.

### Learning Objectives

Through this game, you will discover:
- **Stochastic root-finding**: How to solve $H(\theta) = 0$ when you only observe $H(\theta) + \text{noise}$
- **The Robbins-Monro Algorithm**: The fundamental stochastic approximation method
- **Learning rate decay**: Why shrinking your steps over time leads to convergence
- **The Q-learning connection**: How this exact pattern drives Q-learning in reinforcement learning

Let's tee off! 🏌️

## Setup

Run the three cells below once before starting the exercises.

In [ ]:
# @title Install dependencies - run once {display-mode: "form"}
!pip install numpy matplotlib --quiet

In [ ]:
# @title Game utilities - run once {display-mode: "form"}
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Optional, Tuple


class MiniGolfGame:
    '''
    1D mini-golf putting game with wind noise.

    The hole is hidden. After each putt you receive a signed error:
        positive  = overshot  (ball went past the hole)
        negative  = undershot (ball stopped short)
    '''

    def __init__(self, seed=None, wind_std=8.0, drift_std=0.0):
        self.seed = seed if seed is not None else np.random.randint(0, 2**31)
        self.rng = np.random.RandomState(self.seed)
        self.wind_std = wind_std
        self.drift_std = drift_std
        self._force = 50.0
        self._hole_pos = self._init_hole()
        self.round = 0
        self.history = []

    def _init_hole(self):
        if self.rng.random() < 0.5:
            return 15.0 + self.rng.random() * 15.0
        else:
            return 65.0 + self.rng.random() * 15.0

    def _maybe_drift(self):
        if self.drift_std > 0:
            step = float(np.clip(self.rng.randn() * self.drift_std, -5.0, 5.0))
            self._hole_pos = float(np.clip(self._hole_pos + step, 10.0, 90.0))

    def putt(self, force):
        '''
        Putt with the given force. Wind randomly shifts the landing position.

        Returns a dict with:
            signed_error  - landing minus hole position
            landing       - where the ball ended up
            force         - the force used
            round         - current round number
        '''
        self.round += 1
        self._maybe_drift()
        wind = float(self.rng.randn() * self.wind_std)
        landing = float(np.clip(force + wind, 0.0, 100.0))
        signed_error = landing - self._hole_pos
        result = {
            'round': self.round,
            'force': float(force),
            'landing': landing,
            'signed_error': float(signed_error),
        }
        self.history.append(result)
        return result

    def reset(self):
        '''Reset to the initial state (same seed, same hole position).'''
        self.rng = np.random.RandomState(self.seed)
        self._force = 50.0
        self._hole_pos = self._init_hole()
        self.round = 0
        self.history = []

    def reveal_hole(self):
        '''Return the true hole position (for checking convergence after the exercise).'''
        return self._hole_pos

    @property
    def force(self):
        return self._force

    @force.setter
    def force(self, value):
        self._force = float(np.clip(value, 0.0, 100.0))


def plot_convergence(forces, signed_errors, learning_rates):
    '''Plot force convergence, signed errors, and learning-rate schedule.'''
    window = 10
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(forces, lw=2, color='#22c55e')
    axes[0].set_xlabel('Round')
    axes[0].set_ylabel('Putting force')
    axes[0].set_title('Force estimate over time')
    axes[0].grid(alpha=0.3)

    axes[1].plot(signed_errors, alpha=0.35, color='#f87171', label='Signed error')
    if len(signed_errors) >= window:
        rolling = np.convolve(signed_errors, np.ones(window) / window, mode='valid')
        axes[1].plot(range(window - 1, len(signed_errors)), rolling,
                     lw=2.5, color='#dc2626', label=f'{window}-round avg')
    axes[1].axhline(0, color='#22c55e', ls='--', lw=1.5, label='Target  (error = 0)')
    axes[1].set_xlabel('Round')
    axes[1].set_ylabel('Signed error')
    axes[1].set_title('Overshoot (+) / undershoot (−)')
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    axes[2].plot(learning_rates, lw=2, color='#60a5fa')
    axes[2].set_xlabel('Round')
    axes[2].set_ylabel('α')
    axes[2].set_title('Learning rate schedule')
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


game = MiniGolfGame(seed=42)
print('\u26f3 Mini-Golf utilities loaded!')
print('Ready to find that invisible hole. \U0001f32c\ufe0f')

## Explore the Game

Before writing any code, play the game manually.
Use the **+1 / −1** buttons (or ← → arrow keys) to adjust your putting force, then press **PUTT** (or Space).

Try to zero in on the hole using only the caddie's feedback. How many putts does it take?

In [ ]:
# @title Play Mini-Golf in the Wind - run to launch {display-mode: "form"}
from IPython.display import HTML as _HTML
import urllib.request as _req
import os as _os

try:
    # Local: read the bundled HTML file
    with open('minigolf.html', 'r', encoding='utf-8') as _f:
        _html = _f.read()
except FileNotFoundError:
    # Colab: fetch from the public repo
    _url = 'https://raw.githubusercontent.com/eth-ainit-fs26/coding-exercises/main/minigolf/minigolf.html'
    _html = _req.urlopen(_url).read().decode('utf-8')

_HTML(_html)

### The Optimization Problem

In Mini-Golf in the Wind, there **is** a correct putting force — but:

- You **can't see the hole**
- You **don't know how far it is** from you
- You only get a **signed error** after each putt: how far past or short you went
- The **wind shifts your ball** randomly each time

Your goal is not to get lucky on one putt. It's to **converge to the right force on average**.

Formally, let $\theta$ be your putting force and $\theta^*$ be the force that lands the ball at the hole **on average**. Define the error function:

$$H(\theta) = \mathbb{E}[\text{landing} - \text{hole}] = \theta - \theta^*$$

You want to find $\theta^*$ such that $H(\theta^*) = 0$. This is a **root-finding problem**.

The catch: you can only observe a **noisy sample** of $H(\theta)$ at each step — the wind makes every putt different.

### What "root-finding" means without math

In plain language: find the force where the caddie's feedback **switches from SHORT to PAST**.

- If your force is **too small**: ball lands short → signed error is **negative**
- If your force is **too large**: ball lands past  → signed error is **positive**
- At the **right force**: expected signed error is **zero** — that's the root

The wind means you'll almost never hear exactly *"0.0 feet"* on any single putt. But if you've found the right force, the caddie's feedback will **average out to zero** over many putts.

### Robbins-Monro as a putting strategy

Think of Robbins-Monro as a simple **rule for adjusting your force after each putt**.

#### Putt with your current force
&nbsp;&nbsp;&nbsp;&nbsp;You start somewhere — say, force = 50.

#### Hear the caddie's signed feedback
&nbsp;&nbsp;&nbsp;&nbsp;You get a noisy measurement of how far off you are, **and crucially — in which direction**.

#### Adjust your force
&nbsp;&nbsp;&nbsp;&nbsp;Overshot (+)? Ease off a little. Undershot (−)? Hit a bit harder.

$$\theta_{t+1} = \theta_t - \alpha_t \cdot \underbrace{(\text{landing}_t - \text{hole})}_{\text{noisy signed error } e_t}$$

Over many rounds, this converges to $\theta^*$.

#### Why does this work?
Each signed error $e_t$ is a noisy sample of $H(\theta_t) = \theta_t - \theta^*$.
Subtracting it (scaled by $\alpha_t$) nudges $\theta_t$ toward $\theta^*$.
As long as the steps are large enough to reach the root but small enough that noise averages out, convergence is guaranteed.

## Implement the Robbins-Monro Algorithm

Now it's your turn to implement the Robbins-Monro algorithm for the putting game.

### The Algorithm (fixed learning rate)

For each round $t = 0, 1, 2, \ldots$:

1. **Current force**: use your current guess $\theta_t$ (stored in `game.force`)
2. **Putt**: call `game.putt(θ_t)` → get back a result dict containing `signed_error`
   $$e_t = \text{landing}_t - \text{hole}$$
3. **Update** using the Robbins-Monro rule:
   $$\theta_{t+1} = \theta_t - \alpha \cdot e_t$$

> 💭 *Before you code:* if `e_t = +5` (overshot by 5), should the new force be higher or lower? Does the update formula agree with your intuition?

In [ ]:
def robbins_monro(
    game: MiniGolfGame,
    n_rounds: int = 100,
) -> Tuple[List[float], List[float], List[float]]:
    '''
    Implement the Robbins-Monro root-finding algorithm for Mini-Golf in the Wind.

    Args:
        game:     MiniGolfGame instance
        n_rounds: number of rounds to run

    Returns:
        Tuple of (forces, signed_errors, learning_rates)
    '''
    forces        = []
    signed_errors = []
    learning_rates = []

    game.reset()

    for t in range(n_rounds):
        # Fixed learning rate
        alpha_t = 3.0
        learning_rates.append(alpha_t)

        # -- 1. Record current force ------------------------------------------
        current_force = ...
        # --- 🎯🎯🎯🎯 ---
        forces.append(current_force)

        # -- 2. Putt and observe signed error ---------------------------------
        result = ...
        # --- 🎯🎯🎯🎯 ---
        signed_errors.append(result['signed_error'])

        # -- 3. Extract signed error ------------------------------------------
        e_t = ...
        # --- 🎯🎯🎯🎯 ---

        # -- 4. Robbins-Monro update ------------------------------------------
        # --- 🎯🎯🎯🎯 ---

    return forces, signed_errors, learning_rates

In [ ]:
forces, signed_errors, learning_rates = robbins_monro(game=game, n_rounds=100)

print(f'Starting force:       {forces[0]:.2f}')
print(f'Final force:          {forces[-1]:.2f}')
print(f'True hole position:   {game.reveal_hole():.2f}')
print(f'Final signed error:   {signed_errors[-1]:.2f} ft')
print(f'Avg |error| last 10:  {np.mean(np.abs(signed_errors[-10:])):.2f} ft')

In [ ]:
plot_convergence(forces, signed_errors, learning_rates)

## A Crucial Addition

### Learning Rate Schedule

With a fixed learning rate, the force estimate bounces around the hole without ever fully settling — each round, the wind pushes you off by the same-sized step even when you're already very close.

The fix is to **shrink the learning rate over time**:

$$\alpha_t = \frac{\alpha_0}{1 + t / \text{decay\_rate}}$$

- `α₀` — initial learning rate (how bold your first adjustments are)
- `decay_rate` — controls how quickly the rate shrinks

Why this works:
- **Early rounds**: large steps → move quickly toward the right zone
- **Later rounds**: small steps → fine-tune without overshooting
- **Eventually**: steps so tiny the estimate barely moves → convergence

This is the classic **Robbins-Monro condition**: $\sum_t \alpha_t = \infty$ (enough total movement to reach the root) and $\sum_t \alpha_t^2 < \infty$ (steps small enough that noise averages out).

In [ ]:
def robbins_monro(
    game: MiniGolfGame,
    n_rounds: int = 100,
    alpha_0: float = 5.0,
    decay_rate: float = 20.0,
) -> Tuple[List[float], List[float], List[float]]:
    '''
    Robbins-Monro with a decaying learning rate.

    Args:
        game:       MiniGolfGame instance
        n_rounds:   number of rounds to run
        alpha_0:    initial learning rate
        decay_rate: controls how fast the rate decays

    Returns:
        Tuple of (forces, signed_errors, learning_rates)
    '''
    forces         = []
    signed_errors  = []
    learning_rates = []

    game.reset()

    for t in range(n_rounds):
        # -- Decaying learning rate -------------------------------------------
        # --- 🎯🎯🎯🎯 ---
        learning_rates.append(alpha_t)

        # -- 1. Record current force ------------------------------------------
        # --- 🎯🎯🎯🎯 ---
        forces.append(current_force)

        # -- 2. Putt and observe signed error ---------------------------------
        # --- 🎯🎯🎯🎯 ---
        signed_errors.append(result['signed_error'])

        # -- 3. Extract signed error ------------------------------------------
        # --- 🎯🎯🎯🎯 ---

        # -- 4. Robbins-Monro update ------------------------------------------
        # --- 🎯🎯🎯🎯 ---

    return forces, signed_errors, learning_rates

In [ ]:
forces, signed_errors, learning_rates = robbins_monro(
    game=game,
    n_rounds=100,
    alpha_0=5.0,
    decay_rate=20.0,
)

print(f'Starting force:       {forces[0]:.2f}')
print(f'Final force:          {forces[-1]:.2f}')
print(f'True hole position:   {game.reveal_hole():.2f}')
print(f'Final signed error:   {signed_errors[-1]:.2f} ft')
print(f'Avg |error| last 10:  {np.mean(np.abs(signed_errors[-10:])):.2f} ft')

In [ ]:
plot_convergence(forces, signed_errors, learning_rates)

## From Mini-Golf to Q-Learning

You've just implemented a core algorithm in stochastic optimization. Now connect it to reinforcement learning.

---

**Question 1 — What were you finding?**

In the putting game you searched for a force $\theta^*$ with a specific property. Complete this sentence:

> *"$\theta^*$ is the force where the expected signed error is _____, i.e. $\mathbb{E}[e_t \mid \theta_t = \theta^*] = 0$, because the expected landing equals _____"*

---

**Question 2 — A parallel in RL**

In reinforcement learning we maintain estimates $Q(s, a)$ for every state-action pair. What equation should $Q(s, a)$ satisfy when it has **fully converged**?

> Hint: think Bellman. Write it out.

---

**Question 3 — Rewrite as root-finding**

Define the function:
$$H(Q(s,a)) = Q(s,a) - \mathbb{E}\!\left[r + \gamma \max_{a'} Q(s', a')\right]$$

What value should $H(Q(s,a))$ take at the solution? In other words, finding $Q$ is finding the **root** of which equation?

---

**Question 4 — Identify the noise**

In the putting game, the **wind** was the source of stochasticity — each putt gave a noisy sample of $H(\theta)$ instead of its exact value.

In Q-learning, what plays the role of the wind? Why is each update noisy, even in a fully deterministic environment?

---

**Question 5 — Write the update**

Apply the Robbins-Monro update rule to $Q(s,a)$, using one observed transition $(s, a, r, s')$ as your noisy sample of $H$:

$$Q(s,a) \leftarrow Q(s,a) - \alpha_t \cdot H_{\text{observed}}$$

Expand $H_{\text{observed}}$ and simplify. Write out the resulting update rule. Does it look familiar?